In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

from sklearn.preprocessing import KBinsDiscretizer
from sklearn.compose import ColumnTransformer

In [2]:
df = pd.read_csv('train.csv',usecols=['Age','Fare','Survived'])
df.head()

,Survived,Age,Fare
0,0,22.0,7.2500
1,1,38.0,71.2833
2,1,26.0,7.9250
3,1,35.0,53.1000
4,0,35.0,8.0500


In [3]:
df.dropna(inplace=True)

In [4]:
x = df.iloc[:,1:]
y = df.iloc[:,0]

In [5]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [6]:
x_train.head(2)

,Age,Fare
328,31.0,20.5250
73,26.0,14.4542


In [7]:
clf = DecisionTreeClassifier()
clf.fit(x_train,y_train)
y_pred = clf.predict(x_test)
accuracy_score(y_test,y_pred)

0.6363636363636364

In [8]:
kbin_age = KBinsDiscretizer(n_bins=5,encode='ordinal',strategy='kmeans')
kbin_fare = KBinsDiscretizer(n_bins=5,encode='ordinal',strategy='kmeans')

In [9]:
trf = ColumnTransformer([
    ('first',kbin_age,[0]),
    ('second',kbin_fare,[1])
])

In [10]:
x_train_trf = trf.fit_transform(x_train)
x_test_trf = trf.fit_transform(x_test)

In [11]:
output = pd.DataFrame({
    'age':x_train['Age'],
    'age_trf':x_train_trf[:,0],
    'fare':x_train['Fare'],
    'fare_trf' : x_train_trf[:,1]
})

In [12]:
output['age_labels'] = pd.cut(x=x_train['Age'],bins=trf.named_transformers_['first'].bin_edges_[0].tolist())
output['fare_labels'] = pd.cut(x=x_train['Fare'],bins=trf.named_transformers_['second'].bin_edges_[0].tolist())

In [15]:
output.sample(5)

,age,age_trf,fare,fare_trf,age_labels,fare_labels
532,17.0,1.0,7.2292,0.0,"(13.545, 25.495]","(0.0, 35.744]"
462,47.0,3.0,38.5000,0.0,"(36.147, 48.333]","(35.744, 90.827]"
197,42.0,3.0,8.4042,0.0,"(36.147, 48.333]","(0.0, 35.744]"
675,18.0,1.0,7.7750,0.0,"(13.545, 25.495]","(0.0, 35.744]"
146,27.0,1.0,7.7958,0.0,"(25.495, 36.147]","(0.0, 35.744]"


In [13]:
clf = DecisionTreeClassifier()
clf.fit(x_train,y_train)
y_pred2 = clf.predict(x_test_trf)

d:\Pyhton\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


In [14]:
accuracy_score(y_test,y_pred2)

0.45454545454545453